# Tanglish DistilGPT2 — Fine-Tuning v3 (curated category dataset)

**GPU runtime required.** *Settings → Accelerator → GPU T4 x2 or P100.*
Attach your dataset (`tanglish_train.csv`, `tanglish_validation.csv`) as a Kaggle Dataset input and set `DATASET_NAME` below. Internet is not required this time — everything comes from the attached CSVs.

### What's different in v3, and why

This dataset is small (~460 train / 50 val), clean, and hand-curated with a `category` column (greetings, food, tech_and_work, etc.) — a very different shape from the 185k-row noisy scrape used before. The pipeline is adapted accordingly, and every past mistake is addressed directly:

| Past mistake | Fix in this notebook |
|---|---|
| Block-packed tokenization with no sentence separator | Per-sentence tokenization, EOS appended to every row |
| Epoch-only eval on a large dataset (too sparse) | Here the dataset is *small*, so epoch-level eval is actually the right granularity — but early stopping is added, since a tiny dataset overfits fast |
| transformers v5 broke `TrainingArguments`/`Trainer` kwargs | Training-args/Trainer construction is version-adaptive: detects v5 vs pre-v5 and builds the right kwargs automatically |
| Greedy next-word decoding collapsed onto a handful of generic BPE fragments (`ivan`, `ival`, `iph…`) regardless of context | Two fixes: (1) the tokenizer is extended with the corpus's frequent whole Tanglish words so they aren't forced through ambiguous English BPE splits, and (2) each sentence is trained with an explicit `<cat_X>` category token prefix, giving the model a direct context signal instead of relying purely on inferring domain from a few prior words |
| No defense against overfitting | With only ~460 rows, `EarlyStoppingCallback` on eval_loss is used, plus higher weight decay |


In [ ]:
# ============================================================
#  CONFIGURE THIS CELL BEFORE RUNNING
# ============================================================

DATASET_NAME = "tanglish-curated-data"   # <-- Kaggle input folder name; change me
TRAIN_FILE = "tanglish_train.csv"
VAL_FILE   = "tanglish_validation.csv"

MODEL_NAME = "distilgpt2"

# --- Vocabulary extension ---
MIN_WORD_FREQ_FOR_TOKEN = 3     # a whole word must appear at least this often in train to earn its own token
MAX_EXTRA_WORD_TOKENS   = 60    # cap on how many frequent-word tokens we add

# --- Tokenization ---
MAX_LENGTH = 32   # generous headroom: category-token prefix + up to ~8-word sentences (max observed: 43 chars)

# --- Training ---
NUM_EPOCHS             = 25     # small dataset needs many epochs to accumulate enough gradient steps
PER_DEVICE_TRAIN_BATCH = 8
PER_DEVICE_EVAL_BATCH  = 8
GRAD_ACCUM_STEPS       = 1
LEARNING_RATE          = 5e-5
WARMUP_RATIO           = 0.1
WEIGHT_DECAY           = 0.05   # heavier regularization than the 185k run, given how easy it is to overfit 460 rows
EARLY_STOPPING_PATIENCE = 4     # stop if eval_loss doesn't improve for this many eval rounds
SAVE_TOTAL_LIMIT       = 2

OUTPUT_MODEL_DIR = "/kaggle/working/tanglish-distilgpt2-v3"
OUTPUT_ZIP       = "/kaggle/working/tanglish-distilgpt2-v3.zip"

import os
DATASET_DIR = f"/kaggle/input/{DATASET_NAME}/"
TRAIN_PATH  = os.path.join(DATASET_DIR, TRAIN_FILE)
VAL_PATH    = os.path.join(DATASET_DIR, VAL_FILE)
print(f"Train : {TRAIN_PATH}")
print(f"Val   : {VAL_PATH}")
print(f"Model output: {OUTPUT_MODEL_DIR}")


In [ ]:
# Kaggle already ships torch; only the HF stack needs upgrading
!pip install -q --upgrade transformers datasets accelerate


In [ ]:
import os, re, math, random
from collections import Counter

import pandas as pd
import torch
import transformers
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

set_seed(42)
random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def transformers_major_version() -> int:
    return int(transformers.__version__.split(".")[0])

IS_V5 = transformers_major_version() >= 5

print(f"Device              : {DEVICE}")
print(f"transformers version: {transformers.__version__}  (v5 API: {IS_V5})")
if DEVICE == "cuda":
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Load and validate the CSVs

In [ ]:
def load_split(path: str, name: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{name} file not found: {path}\nCheck DATASET_NAME in the config cell.")
    df = pd.read_csv(path)
    required = {"category", "sentence"}
    if not required.issubset(df.columns):
        raise ValueError(f"{name} CSV must have columns {required}. Found: {list(df.columns)}")
    df["sentence"] = df["sentence"].astype(str).str.strip()
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df = df[df["sentence"].str.len() > 0]
    print(f"{name}: {len(df):,} rows, {df['category'].nunique()} categories")
    return df.reset_index(drop=True)

train_df = load_split(TRAIN_PATH, "train")
val_df   = load_split(VAL_PATH,   "val")

print("\nTrain category distribution:")
print(train_df["category"].value_counts())
print("\nVal category distribution:")
print(val_df["category"].value_counts())

train_only_cats = set(train_df["category"]) - set(val_df["category"])
if train_only_cats:
    print(f"\nNote: categories present in train but not val: {sorted(train_only_cats)}")
    print("This is fine — the category token still gets learned from train rows even without val coverage.")

dup_in_train = train_df["sentence"].duplicated().sum()
overlap = len(set(train_df["sentence"]) & set(val_df["sentence"]))
print(f"\nDuplicate sentences within train: {dup_in_train}")
print(f"Train/val sentence overlap       : {overlap}")


## Vocabulary extension

Two kinds of new tokens are added before training, directly targeting the failure mode observed last time (greedy decoding collapsing onto ambiguous BPE fragments like `ivan`/`ival`/`iph…`):

1. **Category tokens** (`<cat_food>`, `<cat_greetings>`, ...) — one per unique category, giving the model an explicit, unambiguous signal for which domain a sentence belongs to, instead of inferring it purely from a few prior words.
2. **Frequent whole-word tokens** — common Tanglish words that appear often enough in the training set get their own single token, so they don't have to be reconstructed from ambiguous English-BPE sub-word pieces.


In [ ]:
# ── Category tokens ──────────────────────────────────────────
categories = sorted(set(train_df["category"]) | set(val_df["category"]))
category_tokens = [f"<cat_{c}>" for c in categories]
print(f"Category tokens ({len(category_tokens)}): {category_tokens}")

# ── Frequent whole-word tokens ───────────────────────────────
WORD_RE = re.compile(r"[a-zA-Z]+")
word_counts = Counter()
for s in train_df["sentence"]:
    word_counts.update(w.lower() for w in WORD_RE.findall(s))

frequent_words = [w for w, c in word_counts.most_common() if c >= MIN_WORD_FREQ_FOR_TOKEN]
frequent_words = frequent_words[:MAX_EXTRA_WORD_TOKENS]
print(f"\nFrequent whole-word tokens to add ({len(frequent_words)}):")
print(frequent_words)


## Tokenizer — extended with the new tokens

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 family has no pad token — reuse eos_token

vocab_before = len(tokenizer)
tokenizer.add_special_tokens({"additional_special_tokens": category_tokens})
num_word_tokens_added = tokenizer.add_tokens(frequent_words)
vocab_after = len(tokenizer)

print(f"Vocab size before: {vocab_before}")
print(f"Vocab size after : {vocab_after}  (+{vocab_after - vocab_before})")
print(f"EOS token : {tokenizer.eos_token!r} (id {tokenizer.eos_token_id})")
print(f"PAD token : {tokenizer.pad_token!r} (id {tokenizer.pad_token_id})")


## Build category-conditioned training text

Each row becomes `<cat_X> sentence<eos>`. The category token is part of the input the model attends to, not just metadata — it's how the model learns to condition its output on domain.


In [ ]:
def make_text(row) -> str:
    return f"<cat_{row['category']}> {row['sentence']}"

train_df["text"] = train_df.apply(make_text, axis=1)
val_df["text"]   = val_df.apply(make_text, axis=1)

print(train_df[["category", "text"]].head(5).to_string(index=False))

raw_datasets = DatasetDict({
    "train"     : Dataset.from_pandas(train_df[["text"]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["text"]], preserve_index=False),
})
print(raw_datasets)


## Tokenize — per sentence, EOS-separated (no block packing)

Same fix carried over from the previous notebook: tokenize each row individually and append EOS, rather than concatenating everything and chunking into fixed blocks. On a 460-row dataset this matters even more — there's no room to smear sentence boundaries together.


In [ ]:
def tokenize_fn(batch):
    texts_with_eos = [t + tokenizer.eos_token for t in batch["text"]]
    return tokenizer(texts_with_eos, truncation=True, max_length=MAX_LENGTH, padding=False)

tokenized = raw_datasets.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
    desc="Tokenising (per-sentence, EOS-separated)",
)
print(tokenized)

lengths = [len(x) for x in tokenized["train"]["input_ids"]]
print(f"Token-length stats: min={min(lengths)} max={max(lengths)} avg={sum(lengths)/len(lengths):.1f}")
if max(lengths) >= MAX_LENGTH:
    print(f"WARNING: some rows hit the MAX_LENGTH={MAX_LENGTH} cap and were truncated — consider raising it.")


## Model — loaded and resized to the extended vocabulary

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))   # new embedding rows for the tokens we just added
model.config.pad_token_id = tokenizer.pad_token_id

n_params = sum(p.numel() for p in model.parameters())
print(f"DistilGPT2 parameters (after resize): {n_params:,}")


## Training

Version-adaptive `TrainingArguments`/`Trainer` construction — this notebook detects whether it's running on transformers v5+ (which removed `overwrite_output_dir`, `logging_dir`, `group_by_length`, `warmup_ratio`, and `Trainer(tokenizer=...)`) or an earlier version, and builds the right keyword arguments either way, so a Kaggle environment update doesn't silently break the run again.

`EarlyStoppingCallback` is included because 460 rows is genuinely easy for an 82M-parameter model to overfit — better to stop when eval_loss stops improving than to burn through all 25 epochs regardless.


In [ ]:
use_fp16 = DEVICE == "cuda"

args_kwargs = dict(
    output_dir                  = OUTPUT_MODEL_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size  = PER_DEVICE_EVAL_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    lr_scheduler_type           = "cosine",
    fp16                        = use_fp16,
    eval_strategy               = "epoch",   # dataset is small — per-epoch granularity is appropriate here
    save_strategy               = "epoch",
    save_total_limit            = SAVE_TOTAL_LIMIT,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    logging_steps               = 10,
    logging_first_step          = True,
    report_to                   = "none",
)

if IS_V5:
    args_kwargs["warmup_steps"] = WARMUP_RATIO   # v5: warmup_ratio removed, warmup_steps now accepts a float ratio
    # v5 also removed: overwrite_output_dir, logging_dir, group_by_length — simply omitted above
else:
    args_kwargs["warmup_ratio"]         = WARMUP_RATIO
    args_kwargs["overwrite_output_dir"] = True
    args_kwargs["logging_dir"]          = "/kaggle/working/logs"
    args_kwargs["group_by_length"]      = True

training_args = TrainingArguments(**args_kwargs)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer_kwargs = dict(
    model         = model,
    args          = training_args,
    train_dataset = tokenized["train"],
    eval_dataset  = tokenized["validation"],
    data_collator = data_collator,
    callbacks     = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)
trainer_kwargs["processing_class" if IS_V5 else "tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

steps_per_epoch = max(len(tokenized["train"]) // (PER_DEVICE_TRAIN_BATCH * GRAD_ACCUM_STEPS), 1)
print(f"Training on {DEVICE.upper()}  |  fp16={use_fp16}  |  transformers v5 API: {IS_V5}")
print(f"Steps per epoch : {steps_per_epoch}")
print(f"Max total steps : {steps_per_epoch * NUM_EPOCHS}  (may stop earlier via early stopping)")

train_result = trainer.train()

print("\n── Loss / perplexity summary ─────────────────────────────")
for log in trainer.state.log_history:
    if "eval_loss" in log:
        epoch = log.get("epoch", "?")
        ev_loss = log["eval_loss"]
        ppl = math.exp(ev_loss)
        print(f"  Epoch {epoch:>5}  eval_loss={ev_loss:.4f}  perplexity={ppl:.2f}")


In [ ]:
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)
trainer.save_model(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)
print(f"Model saved to: {OUTPUT_MODEL_DIR}")
print("Files:", os.listdir(OUTPUT_MODEL_DIR))


In [ ]:
import zipfile

print(f"Zipping {OUTPUT_MODEL_DIR} -> {OUTPUT_ZIP} ...")
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_MODEL_DIR):
        dirs[:] = [d for d in dirs if not d.startswith("checkpoint-")]
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, os.path.dirname(OUTPUT_MODEL_DIR))
            zf.write(full_path, arcname)

zip_size_mb = os.path.getsize(OUTPUT_ZIP) / 1e6
print(f"Done. ZIP size: {zip_size_mb:.1f} MB")
print(f"Download from: Kaggle Output panel -> {os.path.basename(OUTPUT_ZIP)}")


## Sanity check — category-conditioned next-word prediction

Greedy decoding (the single best next word, matching how the autocomplete feature will actually be used), run across several different categories to check the model is genuinely conditioning on the category token rather than defaulting to the same word regardless of context.


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model     = OUTPUT_MODEL_DIR,
    tokenizer = OUTPUT_MODEL_DIR,
    device    = 0 if DEVICE == "cuda" else -1,
)

SAMPLE_PROMPTS = [
    ("greetings",      "Epdi "),
    ("food",           "Lunch ku "),
    ("tech_and_work",  "Office la "),
    ("commute",        "Bus "),
    ("entertainment",  "Movie "),
    ("family_and_home","Amma "),
]

def next_word_completion(category: str, partial: str, max_tokens: int = 6) -> str:
    prompt = f"<cat_{category}> {partial}"
    out = generator(
        prompt,
        max_new_tokens      = max_tokens,
        do_sample            = False,   # greedy — single best guess, matching real autocomplete usage
        repetition_penalty   = 1.2,
        pad_token_id         = tokenizer.eos_token_id,
    )
    generated  = out[0]["generated_text"]
    completion = generated[len(prompt):].strip()
    next_word  = completion.split(" ")[0] if completion else ""
    return next_word.strip(".,!?;:")

print("── Next-word predictions (category-conditioned) ──────────")
for category, partial in SAMPLE_PROMPTS:
    word = next_word_completion(category, partial)
    print(f"  <{category:<16}> {partial!r:<15} ->  {word!r}")
